# init

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&response_type=code&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly

Enter your authorization code:
··········
Mounted at /content/drive


In [2]:
!pip install transformers

     |████████████████████████████████| 450kB 4.9MB/s 
     |████████████████████████████████| 870kB 16.4MB/s 
     |████████████████████████████████| 1.0MB 27.2MB/s 
  Created wheel for sacremoses: filename=sacremoses-0.0.38-cp36-none-any.whl size=884629 sha256=ce975f2b0322bc93a6d4c379b4159f7ed7dd6158df60aae07cefd818e93355f7
  Stored in directory: /root/.cache/pip/wheels/6d/ec/1a/21b8912e35e02741306f35f66c785f3afe94de754a0eaf1422
Successfully built sacremoses


In [3]:
import torch
torch.__version__

'1.3.1'

In [4]:
if torch.cuda.is_available():
  device = torch.device('cuda')
else:
  device = torch.device('cpu')
device

device(type='cpu')

In [0]:
import os
directory = '/content/drive/My Drive/MY PYTORCH MODELS/Distil RoBERTa (Sequence Classification on CoLA)/my_model_save/'

# Tokenizer

In [6]:
from transformers import RobertaTokenizer
import numpy as np
tokenizer = RobertaTokenizer.from_pretrained(directory)

In [7]:
tokenizer

# Model


In [0]:
from transformers import RobertaForSequenceClassification

In [0]:
model = RobertaForSequenceClassification.from_pretrained(directory)

In [10]:
model.to(device)
'''for param in model.parameters():
    param.requires_grad = False'''
model.eval()
print("model sent to:",device)

model sent to: cpu


# Encoding

In [0]:
query = ["he are a boy", 
         "you are good", 
         "you is bad group", 
         "they are new", 
         "they is new", 
         "I don't know where to go.", 
         "I talked to Winston about Winston",
         "I hope nobody will hurt themselves."
]

In [12]:
len(query)

8

In [0]:
def queryEncoder(sentences): 
  input_ids = []
  for each in sentences:
    #encode will convert sentences to a list of tokens
    #the list of tokens will be converted to list of token ids
    encoded_each = tokenizer.encode(
        text = each,
        add_special_tokens = True
    )
    input_ids.append(np.array(encoded_each))
  return input_ids

In [0]:
input_ids = np.array(queryEncoder(query))

In [15]:
type(input_ids)

numpy.ndarray

In [16]:
input_ids

array([array([   0,  700,   32,   10, 2143,    2]),
       array([   0, 6968,   32,  205,    2]),
       array([   0, 6968,   16, 1099,  333,    2]),
       array([    0, 10010,    32,    92,     2]),
       array([    0, 10010,    16,    92,     2]),
       array([  0, 100, 218,  75, 216, 147,   7, 213,   4,   2]),
       array([    0,   100,  3244,     7, 12415,    59, 12415,     2]),
       array([   0,  100, 1034, 5907,   40, 2581, 1235,    4,    2])],
      dtype=object)

In [17]:
input_ids[0]

array([   0,  700,   32,   10, 2143,    2])

In [18]:
input_ids[0].shape

(6,)

In [19]:
input_ids.shape

(8,)

# Padding

In [0]:
MAX_LEN = 20

In [0]:
def encodedQueryPadding(input_ids):
  padded_input_ids = []
  for index in range(input_ids.shape[0]):
      padded = np.zeros((MAX_LEN,), dtype=np.int64)
      if len(input_ids[index]) < MAX_LEN:
        padded[:len(input_ids[index])] = input_ids[index][:]
        padded_input_ids.append(padded)
      else: 
        padded_input_ids.append(input_ids[index][:MAX_LEN])
  return padded_input_ids

In [22]:
padded_input_ids = np.array(encodedQueryPadding(input_ids))
padded_input_ids.shape

(8, 20)

In [23]:
padded_input_ids[0].shape

(20,)

In [24]:
padded_input_ids[0]

array([   0,  700,   32,   10, 2143,    2,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0])

In [0]:
padded_input_ids = torch.tensor(padded_input_ids)

In [26]:
type(padded_input_ids)

torch.Tensor

# Attention Masking

In [0]:
attention_masks = []

for index in range(padded_input_ids.shape[0]):
  att_mask = [int(each > 0) for each in padded_input_ids[index]]  #1 dim array
  attention_masks.append(att_mask)   #2 dim array

attention_masks = np.array(attention_masks)

In [28]:
attention_masks.shape

(8, 20)

In [29]:
attention_masks[0].shape

(20,)

In [30]:
attention_masks[0]

array([0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [0]:
attention_masks = torch.tensor(attention_masks)

In [32]:
type(attention_masks)

torch.Tensor

# Prediction

In [0]:
predictions = []

In [0]:
with torch.no_grad():
  pred = model(padded_input_ids.to(device), attention_mask=attention_masks.to(device), labels=None)

In [35]:
pred

(tensor([[ 0.6538, -0.2364],
         [-2.2532,  2.5971],
         [ 1.0302, -0.8175],
         [-1.8806,  2.3364],
         [ 1.0065, -0.8335],
         [-2.4182,  2.9319],
         [ 0.9575, -0.7734],
         [-2.4435,  3.1141]]),)

In [36]:
# Get the "logits" output by the model. The "logits" are the output
# values prior to applying an activation function like the softmax.
logits = pred[0]
logits

tensor([[ 0.6538, -0.2364],
        [-2.2532,  2.5971],
        [ 1.0302, -0.8175],
        [-1.8806,  2.3364],
        [ 1.0065, -0.8335],
        [-2.4182,  2.9319],
        [ 0.9575, -0.7734],
        [-2.4435,  3.1141]])

In [0]:
# Move logits to CPU
logits = logits.detach().cpu().numpy()
predictions.append(logits)

In [0]:
predictions = np.array(predictions)

In [39]:
predictions

array([[[ 0.65376425, -0.23643136],
        [-2.2532182 ,  2.5971074 ],
        [ 1.0301774 , -0.81745887],
        [-1.8805981 ,  2.336394  ],
        [ 1.0064831 , -0.83346075],
        [-2.4182208 ,  2.9319453 ],
        [ 0.95752335, -0.7733526 ],
        [-2.4434686 ,  3.114133  ]]], dtype=float32)

In [40]:
predictions[0]

array([[ 0.65376425, -0.23643136],
       [-2.2532182 ,  2.5971074 ],
       [ 1.0301774 , -0.81745887],
       [-1.8805981 ,  2.336394  ],
       [ 1.0064831 , -0.83346075],
       [-2.4182208 ,  2.9319453 ],
       [ 0.95752335, -0.7733526 ],
       [-2.4434686 ,  3.114133  ]], dtype=float32)

In [41]:
sig_pred = torch.sigmoid(torch.tensor(predictions)).numpy()
sig_pred

array([[[0.6578582 , 0.44116598],
        [0.09507223, 0.93067515],
        [0.7369503 , 0.30630332],
        [0.1323202 , 0.91184664],
        [0.73233134, 0.3029138 ],
        [0.08179379, 0.9494032 ],
        [0.7226257 , 0.3157543 ],
        [0.07991749, 0.95747197]]], dtype=float32)

In [42]:
sig_pred[0].shape

(8, 2)

In [0]:
sig_pred_label = []
for i in range(sig_pred.shape[0]):
  sig_pred_label.append(np.argmax(sig_pred[i], axis=1).flatten())

In [44]:
sig_pred_label

[array([0, 1, 0, 1, 0, 1, 0, 1])]